In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

In [2]:
aqi_df = pd.read_csv("../data/List of most-polluted cities by particulate matter concentration.csv")
co2_df = pd.read_csv("../data/co-emissions-per-capita.csv")
renewable_df = pd.read_csv("../data/renewable-share-energy.csv")
death_df = pd.read_csv("../data/death-rate-ambient-air-pollution.csv")

print(aqi_df.shape, co2_df.shape, renewable_df.shape, death_df.shape)

(500, 10) (26509, 4) (6379, 4) (197, 4)


In [3]:
merged_df = pd.merge(aqi_df,co2_df,left_on=["Country","Year"],right_on=["Entity","Year"],how="inner")
print(merged_df.head())
print(merged_df.shape)

   Unnamed: 0  Position Country  City/Town  Year  PM2.5 Temporal coverage  \
0           0         1   India     Kanpur  2016    173              >75%   
1           1         2   India  Faridabad  2016    172              >75%   
2           2         3   India       Gaya  2016    149        50% -< 75%   
3           3         4   India   Varanasi  2016    146              >75%   
4           4         5   India      Patna  2016    144              >75%   

   PM10 Temporal coverage.1  Database version (year) Entity Code  \
0   319                 NaN                     2018  India  IND   
1   316                 NaN                     2018  India  IND   
2   275                 NaN                     2018  India  IND   
3   260                 NaN                     2018  India  IND   
4   266                 NaN                     2018  India  IND   

   CO₂ emissions per capita  
0                  1.750474  
1                  1.750474  
2                  1.750474  
3       

In [4]:
base_df = pd.merge(merged_df, renewable_df, left_on=["Country","Year"], right_on=["Entity","Year"], how="inner")
print(base_df.shape)

(467, 16)


In [5]:
base_df.drop(columns=["Unnamed: 0", "Position", "Temporal coverage", 
                       "Temporal coverage.1", "Database version (year)", 
                       "Entity_x", "Code_x", "Entity_y", "Code_y"], inplace=True)
print(base_df.columns.tolist())
print(base_df.shape)

['Country', 'City/Town', 'Year', 'PM2.5', 'PM10', 'CO₂ emissions per capita', 'Renewables']
(467, 7)


In [6]:
base_df["pm25_normalized"] = (base_df["PM2.5"]/150).clip(0,1.0)
base_df["pm10_normalized"] = (base_df["PM10"]/250).clip(0,1.0)
base_df["co2_normalized"] = (base_df["CO₂ emissions per capita"]/10).clip(0,1.0)
base_df["ren_normalized"] = (1 - ((base_df["Renewables"] - base_df["Renewables"].min()) / (base_df["Renewables"].max() - base_df["Renewables"].min())))

print(base_df["pm25_normalized"].head())
print(base_df["pm10_normalized"].head())
print(base_df["co2_normalized"].head())
print(base_df["ren_normalized"].head())

0    1.000000
1    1.000000
2    0.993333
3    0.973333
4    0.960000
Name: pm25_normalized, dtype: float64
0    1.0
1    1.0
2    1.0
3    1.0
4    1.0
Name: pm10_normalized, dtype: float64
0    0.175047
1    0.175047
2    0.175047
3    0.175047
4    0.175047
Name: co2_normalized, dtype: float64
0    0.750723
1    0.750723
2    0.750723
3    0.750723
4    0.750723
Name: ren_normalized, dtype: float64


In [7]:
base_df["risk_score"] = (((base_df["pm25_normalized"]*0.40) + (base_df["pm10_normalized"]*0.2) + (base_df["co2_normalized"]*0.2) + (base_df["ren_normalized"]*0.2)) * 100)
print(base_df["risk_score"].head())
print(base_df["risk_score"].max())
print(base_df["risk_score"].min())

0    78.515402
1    78.515402
2    78.248735
3    77.448735
4    76.915402
Name: risk_score, dtype: float64
79.4659808943292
17.395951


In [8]:
bins = [0,25,50,75,100]
labels = ["Low Risk","Moderate Risk","High Risk","Very High Risk"]
base_df["risk_category"] = pd.cut(base_df["risk_score"],bins=bins,labels=labels)
print(base_df["risk_category"].head())
print(base_df["risk_category"].value_counts())

0    Very High Risk
1    Very High Risk
2    Very High Risk
3    Very High Risk
4    Very High Risk
Name: risk_category, dtype: category
Categories (4, str): ['Low Risk' < 'Moderate Risk' < 'High Risk' < 'Very High Risk']
risk_category
Moderate Risk     349
High Risk         101
Very High Risk      9
Low Risk            8
Name: count, dtype: int64


In [9]:
base_df.to_csv("../data/base_df.csv", index=False)

In [10]:
death_df = pd.read_csv(r"C:\Users\Shiva\Projects\global-aqi-risk\data\death-rate-ambient-air-pollution.csv")

death_df.rename(columns={"Age-standardized mortality rate attributed to ambient air pollution (deaths per 100,000 population)": "death_rate"}, inplace=True)
print(death_df.shape)
print(death_df.head())


(197, 4)
                Entity Code  Year  death_rate
0          Afghanistan  AFG  2019         143
1              Algeria  DZA  2019          48
2               Angola  AGO  2019          60
3  Antigua and Barbuda  ATG  2019          19
4            Argentina  ARG  2019          29


In [11]:
merged_base_df = pd.merge(base_df,death_df,left_on="Country",right_on="Entity",how="inner")
print(merged_base_df.head())

  Country  City/Town  Year_x  PM2.5  PM10  CO₂ emissions per capita  \
0   India     Kanpur    2016    173   319                  1.750474   
1   India  Faridabad    2016    172   316                  1.750474   
2   India       Gaya    2016    149   275                  1.750474   
3   India   Varanasi    2016    146   260                  1.750474   
4   India      Patna    2016    144   266                  1.750474   

   Renewables  pm25_normalized  pm10_normalized  co2_normalized  \
0    6.369765         1.000000              1.0        0.175047   
1    6.369765         1.000000              1.0        0.175047   
2    6.369765         0.993333              1.0        0.175047   
3    6.369765         0.973333              1.0        0.175047   
4    6.369765         0.960000              1.0        0.175047   

   ren_normalized  risk_score   risk_category Entity Code  Year_y  death_rate  
0        0.750723   78.515402  Very High Risk  India  IND    2019          83  
1        0

In [12]:
merged_base_df.drop(columns=["Entity","Code","Year_y"],inplace=True)
merged_base_df.rename(columns={"Year_x":"Year"},inplace=True)
print(merged_base_df.head())

  Country  City/Town  Year  PM2.5  PM10  CO₂ emissions per capita  Renewables  \
0   India     Kanpur  2016    173   319                  1.750474    6.369765   
1   India  Faridabad  2016    172   316                  1.750474    6.369765   
2   India       Gaya  2016    149   275                  1.750474    6.369765   
3   India   Varanasi  2016    146   260                  1.750474    6.369765   
4   India      Patna  2016    144   266                  1.750474    6.369765   

   pm25_normalized  pm10_normalized  co2_normalized  ren_normalized  \
0         1.000000              1.0        0.175047        0.750723   
1         1.000000              1.0        0.175047        0.750723   
2         0.993333              1.0        0.175047        0.750723   
3         0.973333              1.0        0.175047        0.750723   
4         0.960000              1.0        0.175047        0.750723   

   risk_score   risk_category  death_rate  
0   78.515402  Very High Risk          83 

In [13]:
print(merged_base_df.shape)

(463, 14)


In [14]:
agg_df = merged_base_df.groupby("Country").agg(
    pm25_avg = ("PM2.5","mean"),
    pm10_avg = ("PM10","mean"),
    co2_avg = ("CO₂ emissions per capita","mean"),
    ren_avg = ("Renewables","mean"),
    death_rate = ("death_rate","mean")
)

print(agg_df.head())
print(agg_df.shape)

             pm25_avg    pm10_avg   co2_avg    ren_avg  death_rate
Country                                                           
Bangladesh  71.250000  118.375000  0.485981   0.786563        66.0
Bulgaria    28.333333   42.000000  6.554547  10.402521        63.0
Chile       36.631579   65.157895  4.593699  19.437885        18.0
China       50.374558   85.657244  6.990351  10.833874        64.0
Colombia    35.666667   58.666667  1.950854  25.465654        24.0
(25, 5)


In [15]:
agg_df = agg_df.reset_index()
agg_df.to_csv("../data/agg_df.csv", index=False)
merged_df.to_csv("../data/final_df.csv", index=False)

In [18]:
x = agg_df[["pm25_avg","pm10_avg","co2_avg","ren_avg"]]
y = agg_df[["death_rate"]]

In [19]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test = train_test_split(
    x,y,
    test_size = 0.2,
    random_state = 42
)

print(x_train.shape,x_test.shape,y_train.shape,y_test.shape)

(20, 4) (5, 4) (20, 1) (5, 1)


In [20]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=100,random_state=42)
model.fit(x_train,y_train.values.ravel())
print(model)

RandomForestRegressor(random_state=42)


In [21]:
y_pred = model.predict(x_test)
print(y_pred)
print(y_test)

[52.07 43.78 88.24 34.28 76.14]
    death_rate
8         51.0
16        26.0
0         66.0
23        39.0
11        45.0


In [22]:
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(y_test,y_pred)
print(mae)

15.389999999999997


In [24]:
print(y_test)

    death_rate
8         51.0
16        26.0
0         66.0
23        39.0
11        45.0


In [28]:
prediction_df = pd.DataFrame({
    "Country": agg_df.iloc[y_test.index]["Country"],
    "Actual": y_test.values.ravel(),
    "Predicted": y_pred
})
print(prediction_df)

       Country  Actual  Predicted
8         Iran    51.0      52.07
16        Peru    26.0      43.78
0   Bangladesh    66.0      88.24
23      Turkey    39.0      34.28
11      Kuwait    45.0      76.14
